
# QML-SleepNet — Stage 05 Causal Inference Core — AUDITED v1.2 — Colab-safe LiNGAM

This notebook implements the **Stage-05 components that are independent of Stage 04**.
The guide's **Quantum-Causal Fusion** is intentionally deferred until the Stage-04 quantum outputs are frozen.

## Source-locked Stage-05 components

### Causal Graph Discovery
- PC
- FCI
- NOTEARS
- LiNGAM
- DAG: features → apnea type

### Structural Causal Model
- `X_i = f_i(PA_i, N_i)`
- counterfactual inference
- do-calculus / intervention analysis
- HRV → apnea causal path
- mediation analysis ECG → SpO₂

### Causal Feature Ranking
- Average Causal Effect (ACE)
- interventional distribution
- `P(Y | do(X))` estimation
- causal importance rather than SHAP importance
- confounding-bias adjustment/removal

### Deferred until Stage 04 finishes
The source's Quantum-Causal Fusion block:
- QML output + causal features
- causal-aware attention gate
- intervention-level fusion
- counterfactual augmentation
- causal regularisation loss

Those are **not implemented prematurely** here.

---

# Data/estimation choices forced by the available guide + dataset

The guide states the causal methods but does not prescribe:
- the exact numerical ACE estimator,
- the exact intervention values,
- which individual Stage-03 variables constitute the downstream 16-D causal vector.

To avoid adding a new methodology, this notebook uses a minimal source-aligned instantiation:

1. Start only from **named Stage-03 physiological variables already produced by the guide pipeline**.
2. Separate:
   - a broadly available ECG/HRV causal candidate panel for the all-record 16-D output;
   - the guide-required ECG→SpO₂ / respiratory mediation analysis on the subset where real respiratory/SpO₂ measurements exist.
3. Estimate `P(Y|do(X))` from a multivariable logistic SCM on the training fold.
4. Intervention levels are the training-fold 25th and 75th percentiles.
5. ACE = mean probability difference between those two interventions.
6. Rank by absolute ACE and retain the top 16, exactly matching the Stage-06 `causal features (16-D)` contract.

No SHAP score, classifier accuracy, validation labels, or official x labels is used to choose those 16 causal features.
Graph discovery and ACE ranking are fit on each training fold only.


## v1.1 mixed-variable causal-discovery correction

PC/FCI with Fisher-Z, linear NOTEARS, and DirectLiNGAM are continuous-variable discovery procedures.
The apnea target is binary. Treating that binary target as if it were Gaussian continuous data would weaken
the causal analysis.

Therefore the four **guide-named graph-discovery algorithms are applied to the continuous physiological
feature system only**. The binary apnea node is then represented by the guide's **structural causal outcome
model** (`Y = f(PA_Y,N_Y)`), with its parent candidates and intervention effects estimated by the fold-local
logistic SCM / ACE analysis.

The exported combined DAG records which edges came from discovery and which `feature → apnea` edges are
SCM outcome-parent edges. Nothing is silently presented as algorithmically discovered when it was not.


## v1.2 Colab/Python-3.13 environment correction

The standalone `lingam==1.13.0` package pins `scipy<=1.13.1` because of its `semopy` dependency.
On current Colab Python 3.13 this forces an old SciPy source build and then fails while building `portmin`.

This notebook therefore uses the **DirectLiNGAM implementation already shipped inside `causal-learn`**:
`causallearn.search.FCMBased.lingam.DirectLiNGAM`.

This does **not** change the guide methodology: the Stage-05 algorithm is still DirectLiNGAM.
It only removes an unnecessary incompatible second package stack.

The notebook intentionally does NOT upgrade `pip`, `setuptools`, or `wheel`.


In [ ]:

# Cell 1 — environment and exact project artifacts
!pip -q install --prefer-binary "causal-learn==0.1.4.8"

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import gc, json, math, random, time, hashlib, warnings
import numpy as np
import pandas as pd

from scipy.linalg import expm
from scipy.optimize import minimize

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression

from causallearn.search.ConstraintBased.PC import pc
from causallearn.search.ConstraintBased.FCI import fci
from causallearn.utils.cit import fisherz
from causallearn.search.FCMBased import lingam as cl_lingam

warnings.filterwarnings("ignore")

SEED=42

ROOT=Path("/content/drive/MyDrive/QML_SleepNet")
S3=ROOT/"outputs/GUIDE_EXACT_METRICMAX/03_feature_bank_v1"
BRIDGE=ROOT/"outputs/GUIDE_EXACT_METRICMAX/04_bridge128to8_v1_3_fullgrid"
WINFOLDS=BRIDGE/"winning_folds"

BANK=S3/"guide_stage03_full_feature_bank.npz"
INDEX_CSV=S3/"guide_stage03_target_index.csv"

OUT=ROOT/"outputs/GUIDE_EXACT_METRICMAX/05_causal_core_v1_2"
FOLD_DIR=OUT/"folds"
FINAL_DIR=OUT/"final_fit"
for p in (OUT,FOLD_DIR,FINAL_DIR):
    p.mkdir(parents=True,exist_ok=True)

if not BANK.is_file() or not INDEX_CSV.is_file():
    raise FileNotFoundError("Stage-03 guide feature bank/index missing.")

z=np.load(BANK,allow_pickle=False)
X_ALL=np.asarray(z["X"],np.float32)
FEATURE_NAMES=np.asarray(z["feature_names"]).astype(str)
INDEX=pd.read_csv(INDEX_CSV)

LEARN_MASK=INDEX["record_name"].astype(str).str[0].isin(["a","b","c"]).to_numpy()
TEST_MASK=INDEX["record_name"].astype(str).str.startswith("x").to_numpy()

X_LEARN=X_ALL[LEARN_MASK]
X_TEST=X_ALL[TEST_MASK]

LEARN_INDEX=INDEX.loc[LEARN_MASK].reset_index(drop=True)
TEST_INDEX=INDEX.loc[TEST_MASK].reset_index(drop=True)

Y_LEARN=LEARN_INDEX["y"].to_numpy(np.int8)
UID_LEARN=LEARN_INDEX["uid"].astype(str).to_numpy()
UID_TEST=TEST_INDEX["uid"].astype(str).to_numpy()

# Development rule: discard x-record labels from working metadata immediately.
if "y" in TEST_INDEX.columns:
    TEST_INDEX=TEST_INDEX.drop(columns=["y"])

NAME_TO_COL={n:i for i,n in enumerate(FEATURE_NAMES)}

print("Feature bank:",X_ALL.shape)
print("Learning:",X_LEARN.shape)
print("Official x:",X_TEST.shape)
print("Official x y available in working TEST_INDEX:", "y" in TEST_INDEX.columns)


In [ ]:
# Cell 1B — Colab environment smoke test for the four guide causal methods
import scipy, sklearn, causallearn
from causallearn.search.FCMBased import lingam as cl_lingam

print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("causal-learn import: PASS")
print("DirectLiNGAM source:", cl_lingam.DirectLiNGAM.__module__)

_rng=np.random.default_rng(123)
_x0=_rng.normal(size=300)
_x1=0.8*_x0+_rng.laplace(scale=0.4,size=300)
_x2=-0.5*_x1+_rng.laplace(scale=0.4,size=300)
_X=np.column_stack([_x0,_x1,_x2])

_smoke_lingam=cl_lingam.DirectLiNGAM(random_state=123)
_smoke_lingam.fit(_X)
assert np.asarray(_smoke_lingam.adjacency_matrix_).shape==(3,3)
assert np.isfinite(np.asarray(_smoke_lingam.adjacency_matrix_)).all()
print("DirectLiNGAM functional smoke test: PASS")
del _rng,_x0,_x1,_x2,_X,_smoke_lingam


In [ ]:

# Cell 2 — source-lock audit

SOURCE_LOCK={
    "causal_graph_discovery":["PC","FCI","NOTEARS","LiNGAM"],
    "scm":"X_i = f_i(PA_i,N_i)",
    "counterfactual_inference":True,
    "do_intervention":True,
    "hrv_to_apnea_path":True,
    "mediation":"ECG -> SpO2",
    "causal_ranking":[
        "Average Causal Effect (ACE)",
        "interventional distribution",
        "P(Y|do(X)) estimation",
        "confounding bias adjustment",
    ],
    "stage06_causal_dim":16,
    "quantum_causal_fusion_deferred":True,
    "official_test_labels_used":False,
}
print(json.dumps(SOURCE_LOCK,indent=2))

assert SOURCE_LOCK["causal_graph_discovery"]==["PC","FCI","NOTEARS","LiNGAM"]
assert SOURCE_LOCK["stage06_causal_dim"]==16
assert SOURCE_LOCK["quantum_causal_fusion_deferred"] is True


In [ ]:

# Cell 3 — source-aligned causal variable panels

# Broadly available ECG/HRV candidates used for the all-record 16-D causal representation.
# Every variable below already exists in the guide-built Stage-03 feature bank.
CORE_CANDIDATES=[
    # Time-domain HRV
    "cur_rr_mean_s",
    "cur_sdnn_s",
    "cur_rmssd_s",
    "cur_pnn50",

    # Nonlinear HRV
    "cur_sampen_rr",
    "cur_poincare_sd1_s",
    "cur_poincare_sd2_s",
    "guide_approx_entropy",
    "guide_rqa_recurrence_rate",

    # Frequency-domain HRV
    "lsc_ls_vlf_fraction",
    "lsc_ls_lf_fraction",
    "lsc_ls_hf_fraction",
    "lsc_ls_lf_hf_ratio",

    # ECG / morphology
    "qrs_p2p_median_current",
    "qrs_area_median_current",
    "ramp_signed_median_current",
    "ecg_rms",
    "ecg_spectral_entropy",
    "ecg_median_frequency_hz",
    "guide_qrs_duration_ms",
    "guide_qrs_amplitude",
    "guide_p_wave_amplitude",
    "guide_t_wave_amplitude",
    "guide_st_deviation",
]

# Real respiratory / SpO2 variables are sparse because the source dataset provides them only for a subset.
# They are retained for the explicit guide-required mediation/intervention analysis, but are NOT globally
# median-filled into the all-record 16-D vector.
MULTIMODAL_CAUSAL=[
    "guide_resp_rate_bpm",
    "guide_spo2_nadir",
    "guide_spo2_desaturation_depth",
    "guide_thoraco_abdominal_opposition",
    "guide_ecg_resp_coherence",
    "guide_transfer_entropy_ecg_to_resp",
    "guide_ecg_resp_plv",
    "guide_granger_predictive_index_ecg_to_resp",
    "guide_bivariate_spectral_ratio",
]

missing=[n for n in CORE_CANDIDATES+MULTIMODAL_CAUSAL if n not in NAME_TO_COL]
if missing:
    raise RuntimeError(f"Required guide features missing from Stage-03 bank: {missing}")

core_cols=np.asarray([NAME_TO_COL[n] for n in CORE_CANDIDATES],np.int64)
multi_cols=np.asarray([NAME_TO_COL[n] for n in MULTIMODAL_CAUSAL],np.int64)

coverage=pd.DataFrame({
    "feature":CORE_CANDIDATES+MULTIMODAL_CAUSAL,
    "finite_fraction_learn":[
        float(np.isfinite(X_LEARN[:,NAME_TO_COL[n]]).mean())
        for n in CORE_CANDIDATES+MULTIMODAL_CAUSAL
    ],
})
display(coverage)

if coverage[coverage["feature"].isin(CORE_CANDIDATES)]["finite_fraction_learn"].min()<0.90:
    raise RuntimeError("Unexpected low coverage in the all-record core causal panel.")

coverage.to_csv(OUT/"causal_candidate_coverage.csv",index=False)


In [ ]:

# Cell 4 — exact duplicate-safe five-fold membership

FOLDS={}
for fold in range(5):
    p=WINFOLDS/f"bridge_fold{fold}_data.npz"
    if not p.is_file():
        raise FileNotFoundError(p)
    d=np.load(p,allow_pickle=False)

    tr=np.asarray(d["train_rows"],np.int64)
    va=np.asarray(d["val_rows"],np.int64)

    tr_rec=set(LEARN_INDEX.iloc[tr]["record_name"].astype(str))
    va_rec=set(LEARN_INDEX.iloc[va]["record_name"].astype(str))
    separated=(("c05" in tr_rec and "c06" in va_rec) or
               ("c06" in tr_rec and "c05" in va_rec))
    if separated:
        raise RuntimeError(f"fold {fold}: c05/c06 separation detected")

    FOLDS[fold]={"tr":tr,"va":va}

print("Duplicate-safe fold reuse: PASS")



# NOTEARS implementation note

The guide names NOTEARS but does not prescribe a software package.
To avoid package/version fragility, the notebook uses the original **linear NOTEARS acyclicity formulation**
`h(W)=tr(exp(W∘W))-d` with an augmented-Lagrangian optimization and L1 penalty.

This is an implementation of the named guide algorithm, not a new causal method.


In [ ]:

# Cell 5 — linear NOTEARS

def notears_linear(X,lambda1=0.01,max_iter=50,h_tol=1e-8,rho_max=1e16,w_threshold=0.30):
    X=np.asarray(X,np.float64)
    n,d=X.shape
    X=X-X.mean(axis=0,keepdims=True)

    def _loss(W):
        M=X-X@W
        loss=0.5/n*np.sum(M*M)
        G=-1.0/n*X.T@M
        return loss,G

    def _h(W):
        E=expm(W*W)
        h=np.trace(E)-d
        G=(E.T*W)*2
        return h,G

    def _adj(w):
        return (w[:d*d]-w[d*d:]).reshape(d,d)

    def _func(w,rho,alpha):
        W=_adj(w)
        loss,G_loss=_loss(W)
        h,G_h=_h(W)
        obj=loss+0.5*rho*h*h+alpha*h+lambda1*np.sum(w)
        G_obj=G_loss+(rho*h+alpha)*G_h
        grad=np.concatenate([G_obj.ravel(),-G_obj.ravel()])+lambda1
        return obj,grad

    w_est=np.zeros(2*d*d,dtype=np.float64)
    rho,alpha,h=1.0,0.0,np.inf

    bounds=[]
    for i in range(d):
        for j in range(d):
            if i==j:
                bounds.append((0,0))
            else:
                bounds.append((0,None))
    bounds=bounds+bounds

    for _ in range(max_iter):
        while rho<rho_max:
            sol=minimize(
                _func,w_est,args=(rho,alpha),
                method="L-BFGS-B",jac=True,bounds=bounds
            )
            w_new=sol.x
            h_new=_h(_adj(w_new))[0]
            if h_new>0.25*h:
                rho*=10
            else:
                break

        w_est=w_new
        h=h_new
        alpha+=rho*h

        if h<=h_tol or rho>=rho_max:
            break

    W=_adj(w_est)
    W[np.abs(W)<w_threshold]=0.0
    return W


In [ ]:
# Cell 6 — graph extraction wrappers
# IMPORTANT: graph discovery is performed on continuous physiological features only.
# The binary apnea outcome is attached later through the structural outcome SCM.

def adjacency_from_causallearn_graph(graph_obj):
    if hasattr(graph_obj,"graph"):
        return np.asarray(graph_obj.graph)
    if hasattr(graph_obj,"G") and hasattr(graph_obj.G,"graph"):
        return np.asarray(graph_obj.G.graph)
    raise RuntimeError("Could not extract causal-learn graph matrix.")

def run_graph_discovery(Xstd,seed):
    D=np.asarray(Xstd,np.float64)
    names=CORE_CANDIDATES.copy()

    result={
        "node_names":names,
        "scope":"continuous physiological variables only",
        "binary_apnea_node_handling":"added separately through structural logistic outcome SCM",
    }

    print("  PC...")
    pc_obj=pc(D,alpha=0.05,indep_test=fisherz,stable=True,show_progress=False)
    result["pc"]=adjacency_from_causallearn_graph(pc_obj).tolist()

    print("  FCI...")
    fci_out=fci(
        D,independence_test_method=fisherz,
        alpha=0.05,verbose=False,show_progress=False
    )
    fci_graph=fci_out[0] if isinstance(fci_out,tuple) else fci_out
    result["fci"]=adjacency_from_causallearn_graph(fci_graph).tolist()

    print("  NOTEARS...")
    result["notears"]=notears_linear(D).tolist()

    print("  LiNGAM...")
    lg=cl_lingam.DirectLiNGAM(random_state=seed)
    lg.fit(D)
    result["lingam"]=np.asarray(lg.adjacency_matrix_,float).tolist()

    return result



# SCM, interventions, ACE, and confounding adjustment

The source requests an SCM and `P(Y|do(X))`, but does not prescribe an estimator.

The fold-local implementation is deliberately simple and explicit:

- median imputation and z-scoring are fitted on the **training fold only**;
- one multivariable logistic SCM predicts apnea from all core causal candidates;
- for each candidate feature `X_j`, two counterfactual/interventional copies of the training fold are made:
  - `do(X_j = Q25_j)`
  - `do(X_j = Q75_j)`
- every other candidate stays observed, acting as the adjustment set;
- ACE is the mean difference in predicted apnea probability.

This gives a reproducible g-computation-style interventional estimate without importing another model family.
The causal interpretation remains conditional on the SCM and observational-identification assumptions.


In [ ]:

# Cell 7 — fold-local SCM / ACE ranking / 16-D causal feature construction

def fit_causal_fold(Xtr_raw,ytr,Xv_raw,Xtest_raw=None,seed=42):
    imp=SimpleImputer(strategy="median")
    scaler=StandardScaler()

    Xtr_i=imp.fit_transform(Xtr_raw)
    Xv_i=imp.transform(Xv_raw)
    Xtest_i=None if Xtest_raw is None else imp.transform(Xtest_raw)

    Xtr=scaler.fit_transform(Xtr_i)
    Xv=scaler.transform(Xv_i)
    Xtest=None if Xtest_i is None else scaler.transform(Xtest_i)

    scm=LogisticRegression(
        penalty="l2",C=1.0,max_iter=3000,
        solver="lbfgs",random_state=seed
    )
    scm.fit(Xtr,ytr)

    rows=[]
    for j,name in enumerate(CORE_CANDIDATES):
        q25=float(np.quantile(Xtr[:,j],0.25))
        q75=float(np.quantile(Xtr[:,j],0.75))

        lo=Xtr.copy(); hi=Xtr.copy()
        lo[:,j]=q25
        hi[:,j]=q75

        p_lo=scm.predict_proba(lo)[:,1]
        p_hi=scm.predict_proba(hi)[:,1]

        ace=float(np.mean(p_hi-p_lo))
        rows.append({
            "feature":name,
            "q25_standardized":q25,
            "q75_standardized":q75,
            "p_y1_do_q25":float(p_lo.mean()),
            "p_y1_do_q75":float(p_hi.mean()),
            "ace":ace,
            "abs_ace":abs(ace),
        })

    rank=pd.DataFrame(rows).sort_values(
        ["abs_ace","feature"],ascending=[False,True]
    ).reset_index(drop=True)

    top16=rank.head(16)["feature"].tolist()
    pos=[CORE_CANDIDATES.index(n) for n in top16]

    return {
        "imputer":imp,
        "scaler":scaler,
        "scm":scm,
        "ranking":rank,
        "selected_names":top16,
        "X16_train":Xtr[:,pos].astype(np.float32),
        "X16_val":Xv[:,pos].astype(np.float32),
        "X16_test":None if Xtest is None else Xtest[:,pos].astype(np.float32),
        "Xstd_train":Xtr.astype(np.float32),
    }


In [ ]:

# Cell 8 — guide-required ECG→SpO2 mediation on real multimodal rows only

def mediation_ecg_spo2(train_rows):
    ecg=X_LEARN[train_rows,NAME_TO_COL["ecg_rms"]].astype(float)
    spo2=X_LEARN[train_rows,NAME_TO_COL["guide_spo2_desaturation_depth"]].astype(float)
    y=Y_LEARN[train_rows].astype(int)

    ok=np.isfinite(ecg)&np.isfinite(spo2)
    if ok.sum()<100:
        return {
            "n_real_multimodal_rows":int(ok.sum()),
            "status":"insufficient real ECG-SpO2 support for stable mediation estimate"
        }

    ecg=ecg[ok].reshape(-1,1)
    spo2=spo2[ok]
    y=y[ok]

    # Mediator model: SpO2 desaturation depth ~ ECG RMS
    med=LinearRegression().fit(ecg,spo2)
    a=float(med.coef_[0])

    # Outcome model: apnea ~ ECG RMS + SpO2 desaturation depth
    out=LogisticRegression(max_iter=3000,solver="lbfgs").fit(
        np.column_stack([ecg[:,0],spo2]),y
    )
    b=float(out.coef_[0,1])
    direct_logit=float(out.coef_[0,0])

    return {
        "n_real_multimodal_rows":int(len(y)),
        "mediator_path_a_ecg_to_spo2":a,
        "outcome_path_b_spo2_to_apnea_logit":b,
        "approx_indirect_effect_a_times_b":a*b,
        "direct_ecg_to_apnea_logit_adjusted_for_spo2":direct_logit,
        "status":"regression-based mediation estimate; observational assumptions apply",
    }


In [ ]:

# Cell 9 — five fold-safe causal analyses and 16-D artifacts

FOLD_SUMMARY=[]

for fold in range(5):
    out_npz=FOLD_DIR/f"fold{fold}_causal16.npz"
    rank_csv=FOLD_DIR/f"fold{fold}_ace_ranking.csv"
    graph_json=FOLD_DIR/f"fold{fold}_graphs.json"
    mediation_json=FOLD_DIR/f"fold{fold}_ecg_spo2_mediation.json"

    if out_npz.is_file() and rank_csv.is_file() and graph_json.is_file():
        q=np.load(out_npz,allow_pickle=False)
        selected=np.asarray(q["feature_names"]).astype(str).tolist()
        FOLD_SUMMARY.append({"fold":fold,"selected":selected})
        print("RESUME causal fold",fold)
        continue

    tr=FOLDS[fold]["tr"]
    va=FOLDS[fold]["va"]

    Xtr_raw=X_LEARN[tr][:,core_cols]
    Xva_raw=X_LEARN[va][:,core_cols]

    print(f"\nCAUSAL FOLD {fold} train={len(tr)} val={len(va)}")

    fitted=fit_causal_fold(
        Xtr_raw,Y_LEARN[tr],Xva_raw,
        Xtest_raw=None,seed=SEED+fold
    )

    fitted["ranking"].to_csv(rank_csv,index=False)

    print("Graph discovery...")
    graphs=run_graph_discovery(
        fitted["Xstd_train"],seed=SEED+fold
    )

    # The guide requires a features→apnea DAG. The binary outcome is attached
    # through the structural logistic SCM rather than misusing continuous CI tests.
    scm_parent_edges=[
        {
            "from":name,
            "to":"apnea",
            "edge_source":"structural logistic outcome SCM / ACE ranking",
            "ace":float(
                fitted["ranking"].loc[
                    fitted["ranking"]["feature"]==name,"ace"
                ].iloc[0]
            ),
        }
        for name in fitted["selected_names"]
    ]
    graphs["scm_outcome_parent_edges"]=scm_parent_edges
    graph_json.write_text(json.dumps(graphs,indent=2))

    med=mediation_ecg_spo2(tr)
    mediation_json.write_text(json.dumps(med,indent=2))

    # HRV→apnea path evidence: report HRV entries from the ACE ranking.
    hrv_rank=fitted["ranking"][
        fitted["ranking"]["feature"].str.contains(
            "rr|sdnn|rmssd|pnn|sampen|poincare|lsc|entropy|rqa",
            case=False,regex=True
        )
    ].copy()
    hrv_rank.to_csv(FOLD_DIR/f"fold{fold}_hrv_to_apnea_ace.csv",index=False)

    np.savez_compressed(
        out_npz,
        train_rows=tr,
        val_rows=va,
        y_train=Y_LEARN[tr],
        y_val=Y_LEARN[va],
        feature_names=np.asarray(fitted["selected_names"],dtype="U64"),
        causal16_train=fitted["X16_train"],
        causal16_val=fitted["X16_val"],
    )

    FOLD_SUMMARY.append({
        "fold":fold,
        "selected":fitted["selected_names"]
    })

    print("Top-16 ACE features:",fitted["selected_names"])
    print("Mediation:",med)

pd.DataFrame([
    {"fold":r["fold"],"selected_features":"|".join(r["selected"])}
    for r in FOLD_SUMMARY
]).to_csv(OUT/"fold_causal16_inventory.csv",index=False)



## Multimodal intervention evidence

The respiratory/SpO₂ features have genuine measurements only for the subset of records where those channels exist.
They are therefore **not imputed into every patient and advertised as measured causal variables**.

The next cell computes their observed-support intervention associations on the learning data only.
This preserves the guide's respiratory/SpO₂ causal analysis without allowing sparse imputation to drag down the
all-record 16-D Stage-06 causal representation.


In [ ]:

# Cell 10 — observed-support multimodal causal ranking

multi_rows=[]

for name in MULTIMODAL_CAUSAL:
    x=X_LEARN[:,NAME_TO_COL[name]].astype(float)
    ok=np.isfinite(x)

    if ok.sum()<100:
        multi_rows.append({
            "feature":name,
            "n_observed":int(ok.sum()),
            "status":"insufficient observed support"
        })
        continue

    xv=x[ok].reshape(-1,1)
    y=Y_LEARN[ok]

    med=np.nanmedian(xv)
    xv=np.where(np.isfinite(xv),xv,med)
    sc=StandardScaler().fit_transform(xv)

    clf=LogisticRegression(max_iter=3000,solver="lbfgs").fit(sc,y)

    q25=float(np.quantile(sc[:,0],0.25))
    q75=float(np.quantile(sc[:,0],0.75))

    lo=np.full_like(sc,q25)
    hi=np.full_like(sc,q75)

    p0=clf.predict_proba(lo)[:,1].mean()
    p1=clf.predict_proba(hi)[:,1].mean()

    multi_rows.append({
        "feature":name,
        "n_observed":int(ok.sum()),
        "p_y1_do_q25_association":float(p0),
        "p_y1_do_q75_association":float(p1),
        "ace_association":float(p1-p0),
        "status":"observed-support SCM intervention association",
    })

multi_df=pd.DataFrame(multi_rows)
multi_df.to_csv(OUT/"multimodal_observed_support_interventions.csv",index=False)
display(multi_df)


In [ ]:

# Cell 11 — final all-learning causal ranking and x transformation

final=fit_causal_fold(
    X_LEARN[:,core_cols],
    Y_LEARN,
    X_LEARN[:,core_cols],
    Xtest_raw=X_TEST[:,core_cols],
    seed=SEED+900
)

final["ranking"].to_csv(FINAL_DIR/"final_ace_ranking.csv",index=False)

# Full-learning graphs for final interpretation/reporting.
final_graphs=run_graph_discovery(
    final["Xstd_train"],seed=SEED+900
)
final_graphs["scm_outcome_parent_edges"]=[
    {
        "from":name,
        "to":"apnea",
        "edge_source":"structural logistic outcome SCM / ACE ranking",
        "ace":float(
            final["ranking"].loc[
                final["ranking"]["feature"]==name,"ace"
            ].iloc[0]
        ),
    }
    for name in final["selected_names"]
]
(FINAL_DIR/"final_graphs.json").write_text(json.dumps(final_graphs,indent=2))

final_med=mediation_ecg_spo2(np.arange(len(Y_LEARN),dtype=np.int64))
(FINAL_DIR/"final_ecg_spo2_mediation.json").write_text(
    json.dumps(final_med,indent=2)
)

np.savez_compressed(
    FINAL_DIR/"causal16_for_final_stage06.npz",
    learn_uids=UID_LEARN,
    test_uids=UID_TEST,
    y_learn=Y_LEARN,
    feature_names=np.asarray(final["selected_names"],dtype="U64"),
    causal16_learn=final["X16_train"],
    causal16_test=final["X16_test"],
)

print("Final causal16 names:")
for i,n in enumerate(final["selected_names"],1):
    print(i,n)

print("Final learn causal16:",final["X16_train"].shape)
print("Final x causal16:",final["X16_test"].shape)
print("Official x labels used: NO")


In [ ]:

# Cell 12 — Stage-05 core manifest

manifest={
    "pipeline":"QML-SleepNet Stage05 causal core AUDITED v1.2",
    "graph_discovery":["PC","FCI","NOTEARS","LiNGAM"],
    "graph_discovery_scope":"continuous physiological feature system",
    "binary_apnea_node":"structural logistic outcome SCM; feature→apnea edges are explicitly labelled SCM parent edges",
    "scm":{
        "type":"multivariable logistic structural outcome model",
        "interventions":["training-fold Q25","training-fold Q75"],
        "ace":"mean P(Y=1|do(Q75)) - mean P(Y=1|do(Q25))",
        "confounding_adjustment":"other broadly available source-aligned physiological candidates in the SCM",
        "interpretation":"observational causal interpretation conditional on SCM/identification assumptions",
    },
    "causal16":{
        "selection":"top 16 by absolute training-fold ACE",
        "validation_labels_used_for_selection":False,
        "official_test_labels_used":False,
    },
    "hrv_to_apnea_path":True,
    "ecg_spo2_mediation":True,
    "sparse_multimodal_features_not_globally_imputed_into_causal16":True,
    "quantum_causal_fusion":{
        "status":"DEFERRED",
        "reason":"depends on frozen Stage04 QML outputs",
        "remaining_source_items":[
            "QML output + causal features",
            "causal-aware attention gate",
            "intervention-level fusion",
            "counterfactual augmentation",
            "causal regularisation loss",
        ],
    },
    "fold_artifact_dir":str(FOLD_DIR),
    "final_causal16_path":str(FINAL_DIR/"causal16_for_final_stage06.npz"),
    "next":"after Stage04: complete Quantum-Causal Fusion; combine causal16 with CNN-BiLSTM and QML8 in Stage06",
}
(OUT/"STAGE05_CAUSAL_CORE_MANIFEST.json").write_text(json.dumps(manifest,indent=2))

print("="*110)
print("STAGE05 CAUSAL CORE COMPLETE")
print("="*110)
print("Final causal16:",final["selected_names"])
print("Quantum-Causal Fusion intentionally deferred until Stage04 completes.")
print("NEXT: Stage06 final hybrid only after Stage04 QML + current causal core + temporal branch are ready.")
